# Primera prueba: corresistencia en E. coli de perros

**Datos reales y públicos; proyecto exploratorio, sin uso clínico.**

La pregunta es si existen asociaciones entre resultados de laboratorio que se
mantengan en años posteriores. Usamos muestras de origen urinario de perros y
un panel común de seis antibióticos. La unidad es `Sample Id`: **no podemos
asegurar que las muestras correspondan a animales distintos**.

Este notebook muestra resultados del pipeline ya ejecutado y permite
regenerarlos. No se construyó todavía la aplicación web del proyecto.

Fuentes: [FDA Vet-LIRN](https://www.fda.gov/animal-veterinary/science-research/veterinary-laboratory-investigation-and-response-network),
[archivo oficial 2017–2024](https://www.fda.gov/media/190321/download?attachment) y
[USDA APHIS, informe 2020](https://usbiotechnologyregulation.mrp.usda.gov/sites/default/files/2020_APHIS_AMR_PilotProject_EOY_Report.pdf),
tabla 30, página impresa 35. Los cortes son históricos (CLSI VET01S 2020),
no una guía clínica vigente. La tabla presupone infección urinaria canina;
el archivo de FDA solo permite seleccionar el origen de la muestra.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "experiments/protocolo_primera_prueba.json").is_file()), None)
if root is None:
    raise RuntimeError("Abrir Jupyter desde este proyecto o indicar root manualmente.")
sys.path.insert(0, str(root / "experiments"))
import primera_prueba as trial

# Cambiar a True para leer otra vez el XLSX oficial y regenerar el experimento.
REGENERAR = False
if REGENERAR:
    trial.run()

output = root / "results/primera_prueba"
summary = json.loads((output / "resumen.json").read_text(encoding="utf-8"))
protocol = json.loads((root / "experiments/protocolo_primera_prueba.json").read_text(encoding="utf-8"))
assert summary["protocol"] == protocol, "El protocolo cambió: regenerar resultados."
assert trial.sha256_file(root / protocol["source_file"]) == protocol["source_sha256"]
cohort = pd.read_csv(output / "cohorte.csv", dtype={"sample_id": str})
rules = pd.read_csv(output / "reglas.csv")
prevalence = pd.read_csv(output / "prevalencia.csv")
assert cohort["sample_id"].is_unique
assert len(cohort) == summary["audit"]["counts"]["included_complete_sample_ids"]
print("Datos y protocolo verificados.")
print("Pipeline ejecutado (UTC):", summary["generated_at_utc"])
print("Entorno:", summary["environment"])


Datos y protocolo verificados.
Pipeline ejecutado (UTC): 2026-10-05T06:51:53.380396+00:00
Entorno: {'python': '3.12.14', 'pandas': '2.2.3', 'numpy': '2.3.5'}


## 1. Población y decisiones de limpieza

Seleccionamos E. coli, especie canina, panel COMPGN1F y origen `Urine` o
`Urinary tract`, normalizando espacios y mayúsculas. En 2024 se usa la segunda
etiqueta; su agrupación es una decisión explícita del protocolo.

Excluimos muestras con metadatos contradictorios, ensayos ausentes o MIC que no
permiten decidir si superan el corte R. No contamos cada ensayo de antibiótico
como una muestra nueva. El análisis usa casos completos en los seis fármacos.


In [2]:
labels = {
    "raw_assay_rows": "Ensayos en el archivo oficial",
    "raw_sample_ids": "Muestras en todo el archivo",
    "dog_ecoli_sample_ids": "Muestras caninas de E. coli",
    "urinary_sample_ids": "Origen Urine/Urinary tract, todos los paneles",
    "fixed_panel_sample_ids_before_quality_filter": "Con panel COMPGN1F",
    "excluded_conflicting_sample_ids_in_panel": "Excluidas por metadatos contradictorios",
    "excluded_incomplete_or_indeterminate": "Excluidas por ensayo ausente/MIC ambigua",
    "included_complete_sample_ids": "Muestras finales"
}
counts = summary["audit"]["counts"]
print(pd.DataFrame([{"Etapa": text, "Cantidad": counts[key]} for key, text in labels.items()]).to_string(index=False))
print("\nMuestras finales por año:")
print(cohort.groupby("year").size().rename("n").to_string())
print("\nLaboratorios incluidos:", summary["audit"]["included_laboratories"])
print("\nCobertura antes de exigir casos completos:")
print(pd.DataFrame(summary["audit"]["per_drug"]).to_string(index=False))


                                        Etapa  Cantidad
                Ensayos en el archivo oficial    550795
                  Muestras en todo el archivo     27472
                  Muestras caninas de E. coli      7226
Origen Urine/Urinary tract, todos los paneles      4753
                           Con panel COMPGN1F      4034
      Excluidas por metadatos contradictorios         0
     Excluidas por ensayo ausente/MIC ambigua       101
                             Muestras finales      3933

Muestras finales por año:
year
2017    208
2018    371
2019    502
2020    544
2021    568
2022    585
2023    587
2024    568

Laboratorios incluidos: 25

Cobertura antes de exigir casos completos:
item           drug  tested_samples   R  below_R  indeterminate_tests  absent_tests
 AMI      Amikacina            4034  22     4010                    2             0
 GEN    Gentamicina            4033 135     3898                    0             1
 FOV     Cefovecina            4022 505     

## 2. Qué significa R en esta prueba

**R = MIC inequívocamente igual o superior al corte histórico elegido.**
Por debajo de R incluye resultados intermedios: no significa automáticamente
«susceptible». Solo se aprende la presencia del ítem R, sin inferir eficacia.

La MIC censurada se conserva como intervalo. Por ejemplo, amikacina `<=16`
no permite decidir R; `=16` sí. No sustituimos el operador por una igualdad.
Ampicilina y amoxicilina/clavulánico quedan fuera por falta de cortes I/R en
la tabla elegida. No atribuimos mecanismos genéticos como BLEE.


In [3]:
thresholds = pd.DataFrame([
    {"Código": c, "Antibiótico": d["spanish"], "Familia": d["class"], "Corte R (µg/mL)": d["resistant_min"]}
    for c, d in protocol["drugs"].items()
])
print(thresholds.to_string(index=False))
print("\nCasos de borde comprobados:", trial.verify_interval_handling())
examples = [("<=", 16, 16), ("<", 16, 16), ("=", 16, 16), (">", 8, 16)]
for sign, mic, threshold in examples:
    result = trial.resistance_status(sign, mic, threshold)
    meaning = {1: "R", 0: "por debajo de R", None: "indeterminado"}[result]
    print(f"MIC {sign}{mic}, corte R >= {threshold}: {meaning}")


Código    Antibiótico          Familia  Corte R (µg/mL)
   AMI      Amikacina  Aminoglucósidos               16
   GEN    Gentamicina  Aminoglucósidos                8
   FOV     Cefovecina   Cefalosporinas                8
   CPD    Cefpodoxima   Cefalosporinas                8
   EFX  Enrofloxacino Fluoroquinolonas                4
   MAR Marbofloxacino Fluoroquinolonas                4

Casos de borde comprobados: 10
MIC <=16, corte R >= 16: indeterminado
MIC <16, corte R >= 16: por debajo de R
MIC =16, corte R >= 16: R
MIC >8, corte R >= 16: indeterminado


## 3. Descubrimiento y validación temporal

Se buscan reglas con Apriori en **2017–2022**, con un máximo de dos ítems en
el antecedente y uno en el consecuente. Se miden sin reentrenar en **2023–2024**.
Los mismos `Sample Id` no aparecen en ambos períodos. Esto no impide que un
animal no identificable contribuya más de una muestra.

Comparamos dos configuraciones de soporte/confianza definidas en el protocolo.
Ambas requieren lift >=1,2 y al menos 30 coincidencias en descubrimiento. No
seleccionamos una configuración ganadora por los resultados posteriores.

- **Soporte:** fracción de todas las muestras con antecedente y consecuente.
- **Confianza:** fracción de muestras con el antecedente que también tienen el consecuente.
- **Lift:** confianza dividida por la frecuencia del consecuente en ese período.

Son métricas de asociación; no son precisión de un diagnóstico.


In [4]:
discovery = cohort[cohort["year"].isin(protocol["cohort"]["discovery_years"])]
validation = cohort[cohort["year"].isin(protocol["cohort"]["validation_years"])]
assert set(discovery["sample_id"]).isdisjoint(validation["sample_id"])
print("Descubrimiento:", len(discovery), "| Validación:", len(validation))
comparison = pd.DataFrame([
    {k: v for k, v in c.items() if k != "apriori_candidates"}
    for c in summary["comparison"]
])
print(comparison.to_string(index=False))
unique_counts = rules.groupby("rule")["configuration"].nunique()
print("\nReglas distintas:", rules["rule"].nunique())
print("Reglas presentes en ambas configuraciones:", int(unique_counts.eq(2).sum()))
print("\nFrecuencia de R en la cohorte seleccionada, no prevalencia poblacional:")
table = prevalence[["drug", "discovery_R_count", "validation_R_count"]].copy()
table["R % 2017–2022"] = (prevalence["discovery_R_rate"] * 100).round(2)
table["R % 2023–2024"] = (prevalence["validation_R_rate"] * 100).round(2)
print(table.to_string(index=False))


Descubrimiento: 2778 | Validación: 1155
configuration  min_support  min_confidence  discovery_n  validation_n  rules_found  same_class_only_rules  strictly_cross_class_rules  passes_all_thresholds_in_validation
 exploratoria         0.02             0.6         2778          1155           12                      4                           0                                   12
     estricta         0.05             0.8         2778          1155           12                      4                           0                                   12

Reglas distintas: 12
Reglas presentes en ambas configuraciones: 12

Frecuencia de R en la cohorte seleccionada, no prevalencia poblacional:
          drug  discovery_R_count  validation_R_count  R % 2017–2022  R % 2023–2024
     Amikacina                 15                   6           0.54           0.52
   Gentamicina                 93                  37           3.35           3.20
    Cefovecina                340                 156 

## 4. Reglas y su posible redundancia

Una regla con tres fármacos puede parecer cruzar familias y aun así repetir
la relación entre dos de la misma familia. Por eso marcamos si el antecedente
ya incluye un fármaco de la familia del consecuente. Consideramos estrictamente
entre familias solo las reglas sin esa relación.

Todas las reglas de esta ejecución tienen esa relación de familia. La ausencia
de otras reglas significa que no pasan estos umbrales, no que no exista ninguna
asociación en los datos.


In [5]:
chosen = rules[rules["configuration"].eq("exploratoria")].copy()
table = chosen[["rule", "same_class_only", "strictly_cross_class", "discovery_joint_count", "validation_joint_count"]].copy()
table["Confianza % descubrimiento"] = (chosen["discovery_confidence"] * 100).round(2)
table["Confianza % validación"] = (chosen["validation_confidence"] * 100).round(2)
table["Lift validación"] = chosen["validation_lift"].round(2)
print(table.to_string(index=False))
print("\nReglas estrictamente entre familias:", int(chosen["strictly_cross_class"].sum()))


            rule  same_class_only  strictly_cross_class  discovery_joint_count  validation_joint_count  Confianza % descubrimiento  Confianza % validación  Lift validación
FOV + MAR -> EFX            False                 False                    151                      66                      100.00                  100.00            10.90
FOV + EFX -> MAR            False                 False                    151                      66                      100.00                   98.51            10.73
CPD + EFX -> FOV            False                 False                    150                      67                      100.00                  100.00             7.40
CPD + MAR -> FOV            False                 False                    150                      66                      100.00                  100.00             7.40
CPD + MAR -> EFX            False                 False                    150                      66                      100.00          

In [6]:
# Ejemplos fáciles de interpretar, sin seleccionar un tratamiento.
for antecedent, consequent in [("EFX", "MAR"), ("FOV", "CPD")]:
    row = chosen[(chosen["antecedent"] == antecedent) & (chosen["consequent"] == consequent)].iloc[0]
    a = protocol["drugs"][antecedent]["spanish"]
    b = protocol["drugs"][consequent]["spanish"]
    print(f"\nR en {a} -> R en {b} (según cortes del protocolo)")
    print(f"2023–2024: {int(row['validation_joint_count'])}/{int(row['validation_antecedent_count'])} muestras con el antecedente.")
    print(f"Confianza: {row['validation_confidence']:.1%}; soporte: {row['validation_support']:.1%}; lift: {row['validation_lift']:.2f}.")
    print(f"Frecuencia basal del consecuente: {row['validation_baseline']:.1%}.")
    print(f"Intervalo Wilson descriptivo 95% de confianza: {row['validation_confidence_ci95_low']:.1%}–{row['validation_confidence_ci95_high']:.1%}.")

# Contraste independiente de los recuentos de validación de todas las reglas.
for _, row in chosen.iterrows():
    a_ids = set(validation["sample_id"])
    for item in row["antecedent"].split("|"):
        a_ids &= set(validation.loc[validation["R_" + item].eq(1), "sample_id"])
    b_ids = set(validation.loc[validation["R_" + row["consequent"]].eq(1), "sample_id"])
    assert len(a_ids) == row["validation_antecedent_count"]
    assert len(a_ids & b_ids) == row["validation_joint_count"]
print("\nRecuentos verificados por intersección de identificadores.")



R en Enrofloxacino -> R en Marbofloxacino (según cortes del protocolo)
2023–2024: 105/106 muestras con el antecedente.
Confianza: 99.1%; soporte: 9.1%; lift: 10.79.
Frecuencia basal del consecuente: 9.2%.
Intervalo Wilson descriptivo 95% de confianza: 94.8%–99.8%.

R en Cefovecina -> R en Cefpodoxima (según cortes del protocolo)
2023–2024: 154/156 muestras con el antecedente.
Confianza: 98.7%; soporte: 13.3%; lift: 6.91.
Frecuencia basal del consecuente: 14.3%.
Intervalo Wilson descriptivo 95% de confianza: 95.4%–99.6%.

Recuentos verificados por intersección de identificadores.


## 5. Qué podemos concluir

**La prueba de factibilidad funciona:** hay suficientes muestras para un
experimento acotado, se manejan los resultados censurados y se obtienen reglas
que también aparecen en años posteriores. Las dos configuraciones producen
las mismas 12 reglas, por lo que estos umbrales todavía no diferencian alternativas.

**La utilidad adicional para CliniPets sigue abierta:** los hallazgos fuertes
de esta prueba incluyen fármacos de la misma familia y pueden reflejar relaciones
ya conocidas. Una confianza alta por sí sola no prueba novedad ni utilidad clínica.

Un alcance justificable sería un explorador educativo de corresistencia con
recuentos y procedencia. El siguiente experimento deberá estudiar reglas menos
redundantes, sensibilidad a paneles y umbrales, y estabilidad por laboratorio.
Si se modifican parámetros tras mirar esta validación, ese trabajo será
exploratorio y requerirá una evaluación nueva para afirmar generalización.

El dato es de vigilancia en EE. UU.; no estima lo que ocurre en Chile. No hay
tratamientos, desenlaces, dosis ni identificación de pacientes para predecir
qué antibiótico funcionará. La selección por casos completos puede introducir
sesgo. Los intervalos mostrados no corrigen dependencia entre muestras ni
búsqueda de múltiples reglas.

## Reproducción y archivos

Desde la raíz del proyecto:

```bash
python -m pip install -r requirements.txt
python experiments/primera_prueba.py
python experiments/generar_notebook.py
```

Se verifica el SHA-256 del archivo oficial antes de analizarlo. El protocolo
incluye fuentes, cortes, cohortes y configuraciones. Los CSV en
`results/primera_prueba/` conservan la cohorte, ensayos interpretados, auditoría,
frecuencias y reglas. `resumen.json` registra parámetros, entorno y controles.
El script de generación ejecuta estas celdas Python con captura de stdout;
no necesita un kernel Jupyter para generar este archivo.
